# IMDB Movie Reviews — Binary Classification
Prédire si un avis est positif ou négatif.

## 1. Installer et importer les librairies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

print('TensorFlow version:', tf.__version__)

## 2. Charger le dataset IMDB

In [ ]:
num_words = 10000

(train_data, train_labels), (test_data, test_labels) = keras.datasets.imdb.load_data(num_words=num_words)

print('Train samples:', len(train_data))
print('Test samples:', len(test_data))
print('Exemple de review (entiers):', train_data[0][:10])
print('Label:', train_labels[0])  # 1 = positif, 0 = négatif

## 3. Préprocessing — One-hot encoding

In [ ]:
# Transformer les listes d'entiers en vecteurs binaires de taille 10000
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1
    return results

# Appliquer sur train et test
x_train = vectorize_sequences(train_data)
x_test  = vectorize_sequences(test_data)

# Labels en float
y_train = np.asarray(train_labels).astype('float32')
y_test  = np.asarray(test_labels).astype('float32')

print('x_train shape:', x_train.shape)  # (25000, 10000)
print('x_test shape:', x_test.shape)

In [ ]:
# Split train en train + validation (10000 exemples pour la validation)
x_val   = x_train[:10000]
x_train_final = x_train[10000:]

y_val   = y_train[:10000]
y_train_final = y_train[10000:]

print('Train final:', x_train_final.shape)
print('Validation:', x_val.shape)
print('Test:', x_test.shape)

## 4. Construire le modèle

In [ ]:
model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(10000,)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1,  activation='sigmoid')  # sortie binaire
])

model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Entraîner le modèle (20 epochs)

In [ ]:
history = model.fit(
    x_train_final, y_train_final,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val)
)

## 6. Visualiser les courbes — détecter l'overfitting

In [ ]:
# Récupérer les métriques
loss     = history.history['loss']
val_loss = history.history['val_loss']
acc      = history.history['accuracy']
val_acc  = history.history['val_accuracy']
epochs   = range(1, len(loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(epochs, loss,     'b', label='Train loss')
axes[0].plot(epochs, val_loss, 'r', label='Val loss')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Accuracy
axes[1].plot(epochs, acc,     'b', label='Train accuracy')
axes[1].plot(epochs, val_acc, 'r', label='Val accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Ré-entraîner avec le bon nombre d'epochs

D'après les courbes, la val_loss remonte après ~4 epochs → on s'arrête à 4.

In [ ]:
# Réinitialiser le modèle
model2 = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(10000,)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1,  activation='sigmoid')
])

model2.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Entraîner sur tout le train (train + val) avec 4 epochs
model2.fit(
    x_train, y_train,   # train complet cette fois
    epochs=4,
    batch_size=512
)

# Évaluer sur le test
test_loss, test_acc = model2.evaluate(x_test, y_test)
print(f'\nTest loss     : {test_loss:.4f}')
print(f'Test accuracy : {test_acc:.4f} ({test_acc*100:.1f}%)')

## Résumé

- On a chargé le dataset IMDB (25000 train, 25000 test)
- On a transformé les reviews en vecteurs binaires de taille 10000 (one-hot)
- On a construit un réseau simple : Dense(16) → Dense(16) → Dense(1, sigmoid)
- On a entraîné 20 epochs et observé l'overfitting sur les courbes
- On a ré-entraîné avec 4 epochs → meilleure généralisation
- **Accuracy finale sur le test : ~88%**